# Einfache Regression: Kraftstoffeffizienz vorhersagen

*Dieses Notebook basiert auf dem [Tutorial-Notebook](https://www.tensorflow.org/tutorials/keras/regression) von TensorFlow.*

---
Bei einem *Regressionsproblem* wollen wir einen stetigen Wert vorhersagen, etwa einen Preis oder eine Wahrscheinlichkeit.

Dieses Notebook nutzt den klassischen Datensatz [Auto MPG](https://archive.ics.uci.edu/ml/datasets/auto+mpg), den du schon aus früheren Notebooks kennst, und erstellt ein Modell, das die Kraftstoffeffizienz von Autos aus den späten 1970er- und frühen 1980er-Jahren vorhersagt. Dafür geben wir dem Modell eine Beschreibung vieler Autos aus dieser Zeit. Diese Beschreibung enthält Attribute wie Zylinder, Hubraum, PS und Gewicht.

Dieses Beispiel nutzt die `tf.keras`-API. Details findest du in [diesem Leitfaden](https://www.tensorflow.org/guide/keras).

## Setup

In [ ]:
import time
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Ausgaben von numpy leichter lesbar machen.
np.set_printoptions(precision=3, suppress=True)

In [ ]:
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
import tensorflow.keras.layers as kl

print(tf.__version__)

from tensorflow.keras.optimizers import Adam

### Setup für TensorBoard

Wir verwenden das TensorBoard, um einige Ergebnisse zu visualisieren. Mehr Informationen und das TensorBoard selbst findest du am Ende dieses Notebooks. Den Pfad, unter dem die Informationen gespeichert werden sollen, legen wir aber gleich hier am Anfang fest. 

In [ ]:
# Mit diesem Befehl kannst du alle Logs aus früheren Durchläufen löschen
# Wenn du verschiedene Durchläufe vergleichen willst, kannst du diese Zelle überspringen
!rm -rf my_logs/

In [ ]:
# Pfad für das neue Verzeichnis festlegen
root_logdir = os.path.join(os.curdir, "my_logs")

In [ ]:
# Funktion definieren, die für jeden Durchlauf einen neuen Ordner erstellt
def get_run_logdir():
    run_id = time.strftime("run_%d_%m_%Y-%H_%M_%S")
    return os.path.join(root_logdir, run_id)

In [ ]:
run_logdir = get_run_logdir()

In [ ]:
# Funktion für die Verwendung von Callbacks erstellen; "name" sollte der Name des verwendeten Modells sein
def get_callbacks(name):
    return tf.keras.callbacks.TensorBoard(run_logdir + name, histogram_freq=1)

## Der Datensatz Auto MPG

Der Datensatz ist im [UCI Machine Learning Repository](http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data) verfügbar.


### Daten abrufen
Lade zuerst den Datensatz herunter und importiere ihn mit pandas:

In [ ]:
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

column_names = [
    "MPG",
    "Cylinders",
    "Displacement",
    "Horsepower",
    "Weight",
    "Acceleration",
    "Model Year",
    "Origin",
]

raw_dataset = pd.read_csv(
    url, names=column_names, na_values="?", comment="\t", sep=" ", skipinitialspace=True
)

In [ ]:
df = raw_dataset.copy()
df.head()

### Daten bereinigen

Die Spalte `Horsepower` enthält ein paar unbekannte Werte.

In [ ]:
df.isna().sum()

Wir entfernen diese Zeilen, damit dieses Tutorial einfach bleibt.

In [ ]:
df = df.dropna()

Prüfen wir die Datentypen und die Anzahl eindeutiger Werte in jeder Spalte.

In [ ]:
df.info()

In [ ]:
df.nunique()

Die Spalte `"Origin"` ist eigentlich kategorial, nicht numerisch. Deshalb müssen wir sie in Dummy-Variablen umwandeln:

(Hinweis: Du kannst das `keras.Model` so einrichten, dass es diese Art von Transformation für dich übernimmt. Das geht über dieses Tutorial hinaus. Beispiele findest du in den Tutorials [Vorverarbeitungsschichten](https://www.tensorflow.org/tutorials/structured_data/preprocessing_layers) und [Loading CSV data](https://www.tensorflow.org/tutorials/load_data/csv).)

In [ ]:
df["Origin"] = df["Origin"].map({1: "USA", 2: "Europe", 3: "Japan"})

In [ ]:
df = pd.get_dummies(df, prefix="", prefix_sep="", dtype="uint8")
df.head()

### Daten in Trainings- und Testdaten aufteilen

Jetzt teilen wir den Datensatz in einen Trainingsdatensatz und einen Testdatensatz auf.

Den Testdatensatz verwenden wir bei der abschließenden Evaluation unserer Modelle.

In [ ]:
df_train = df.sample(frac=0.8, random_state=0)
df_test = df.drop(df_train.index)

### Daten untersuchen

Werfen wir einen kurzen Blick auf die gemeinsame Verteilung einiger Spaltenpaare aus dem Trainingsdatensatz.

In der obersten Zeile sollte klar zu sehen sein, dass die Kraftstoffeffizienz (MPG) eine Funktion aller anderen Parameter ist. In den anderen Zeilen sollte klar zu sehen sein, dass sie jeweils Funktionen voneinander sind.

In [ ]:
sns.pairplot(df_train[["MPG", "Cylinders", "Displacement", "Weight"]], diag_kind="kde");

Sieh dir auch die allgemeinen Kennzahlen an und achte darauf, wie unterschiedlich die Wertebereiche der einzelnen Features sind:

In [ ]:
df_train.describe().transpose()

### Features und Labels trennen

Bevor wir mit der Modellierung beginnen können, müssen wir unser Label vom Datensatz trennen. Das Label ist der Wert, für dessen Vorhersage wir das Modell trainieren.

In [ ]:
X_train = df_train.copy()
X_test = df_test.copy()

y_train = X_train.pop("MPG")
y_test = X_test.pop("MPG")

## Sklearn

Bevor wir TensorFlow und Keras verwenden, trainieren wir zum Vergleich ein einfaches `LinearRegression`-Modell aus sklearn. 

In [ ]:
# Daten skalieren
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Modell initialisieren und trainieren
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)

In [ ]:
# Vorhersagen treffen
y_pred = lin_reg.predict(X_test_scaled)

In [ ]:
# Modell evaluieren
mae = round(mean_absolute_error(y_test, y_pred), 2)
mse = round(mean_squared_error(y_test, y_pred), 2)

print("MAE:", mae)
print("MSE", mse)

Wir speichern das Ergebnis in einem Dictionary, um am Ende die Ergebnisse der verschiedenen Modelle zu vergleichen.

In [ ]:
test_results = {}
test_results["sklearn_model"] = [mae, mse]

## Normalisierung

In der Tabelle mit den Kennzahlen ist gut zu sehen, wie unterschiedlich die Wertebereiche der einzelnen Features sind.

In [ ]:
X_train.describe().transpose()[["mean", "std"]]

Es hat sich bewährt, Features mit unterschiedlichen Skalen und Wertebereichen zu normalisieren.

Das ist unter anderem wichtig, weil die Features mit den Gewichten des Modells multipliziert werden. Die Skala der Ausgaben und die Skala der Gradienten hängen also von der Skala der Eingaben ab.

Ein Modell *kann* zwar auch ohne Normalisierung der Features konvergieren, aber die Normalisierung macht das Training viel stabiler. 

### Die Normalisierungsschicht
Die `preprocessing.Normalization`-Schicht ist ein sauberer und einfacher Weg, diese Vorverarbeitung in dein Modell einzubauen.

Im ersten Schritt erstellst du die Schicht:

In [ ]:
normalizer = kl.Normalization()

Pass sie dann mit `.adapt()` an die Daten an:

In [ ]:
X_train

In [ ]:
normalizer.adapt(X_train.values)  # adapt erwartet ein Array

Das berechnet den Mittelwert und die Varianz und speichert beide in der Schicht. 

In [ ]:
normalizer.mean.numpy()

Wenn die Schicht aufgerufen wird, gibt sie die Eingabedaten zurück, wobei jedes Feature unabhängig normalisiert ist. Wir können uns die erste Beobachtung im Trainingsdatensatz ansehen und die ursprünglichen mit den normalisierten Features vergleichen:

In [ ]:
first = X_train[:1] * 1.0

with np.printoptions(precision=2, suppress=True):
    print("First example:", first)
    print()
    print("Normalized:", normalizer(first.values).numpy())

## Lineare Regression

Bevor wir ein DNN-Modell (tiefes neuronales Netz) erstellen, beginnen wir mit einer linearen Regression.

### Eine Variable

Wir beginnen mit etwas Einfachem, einer linearen Regression mit einer einzigen Variable, um `MPG` aus `Horsepower` vorherzusagen.

Das Training eines Modells mit `tf.keras` beginnt meist damit, die Modellarchitektur zu definieren.

Hier verwenden wir ein `keras.Sequential`-Modell. Dieses Modell stellt eine Abfolge von Schritten dar. In diesem Fall sind es zwei Schritte:

1. Die Eingabe `horsepower` normalisieren.
2. Mit `layers.Dense` eine lineare Transformation ($y = mx+b$) anwenden, die 1 Ausgabe erzeugt.

Leg für dieses Modell mit nur einem Feature die Anzahl der _Eingaben_ mit einer `keras.Input(shape=(1,))`-Schicht fest oder lass Keras sie beim ersten Aufruf des Modells ableiten.

1. Erstell zuerst die `Normalization`-Schicht für die PS:

In [ ]:
kl.Input(
    shape=[
        1,
    ]
)

In [ ]:
horsepower = np.array(X_train["Horsepower"])  # entspricht X_train['Horsepower'].values

horsepower_normalizer = kl.Normalization(axis=None)
horsepower_normalizer.adapt(horsepower)

2. Dann erstellen wir das Sequential-Modell:

In [ ]:
horsepower_model = tf.keras.Sequential(
    [keras.Input(shape=(1,)), horsepower_normalizer, layers.Dense(units=1)]
)

# Mit der folgenden Zeile können wir eine Zusammenfassung der Modellarchitektur ausgeben:
horsepower_model.summary()

Dieses Modell sagt `MPG` aus `Horsepower` vorher.

Führ das untrainierte Modell auf den ersten 10 PS-Werten aus. Die Ausgabe wird nicht gut sein, aber du siehst, dass sie die erwartete Form `(10,1)` hat:

In [ ]:
horsepower_model.predict(horsepower[:10]).shape

In [ ]:
horsepower_model.predict(horsepower[:10])

Sobald das Modell erstellt ist, konfiguriere das Trainingsverfahren mit der Methode `Model.compile()`. Die wichtigsten Argumente beim Kompilieren sind `loss` und `optimizer`, denn sie legen fest, was optimiert wird (`mean_absolute_error`) und wie (mit `optimizers.Adam`). Außerdem legen wir die `metrics` so fest, dass `mean_squared_error` verwendet wird.

In [ ]:
horsepower_model.compile(
    optimizer=Adam(learning_rate=0.1), loss=["mae"], metrics=["mse"]
)

Sobald das Training konfiguriert ist, führ das Training mit `Model.fit()` aus:

In [ ]:
%%time
history = horsepower_model.fit(
    df_train["Horsepower"],
    y_train,
    epochs=100,
    # Logging unterdrücken (wenn du die Ausgabe für die einzelnen Epochen sehen willst, setz den Wert auf 1)
    verbose=0,
    # Validierungsergebnisse auf 20% der Trainingsdaten berechnen
    validation_split=0.2,
    # Informationen für TensorBoard speichern
    callbacks=get_callbacks("horsepower_model"),
)

Visualisiere den Trainingsfortschritt des Modells mit den Kennzahlen, die im `history`-Objekt gespeichert sind.

In [ ]:
hist = pd.DataFrame(history.history)
hist["epoch"] = history.epoch

# Ergebnisse der ersten 5 Epochen anzeigen
hist.head()

In [ ]:
# Ergebnisse der letzten 5 Epochen anzeigen (loss und val_loss sind gesunken)
hist.tail()

In [ ]:
def plot_loss(history):
    plt.plot(history.history["loss"], label="loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.ylim([0, 10])
    plt.xlabel("Epoch")
    plt.ylabel("Error [MPG]")
    plt.legend()
    plt.grid(True)

In [ ]:
plot_loss(history)

Wir speichern die Ergebnisse auf dem Testdatensatz für später:

In [ ]:
test_results["horsepower_model"] = horsepower_model.evaluate(
    X_test["Horsepower"], y_test, verbose=0
)

Da es eine Regression mit einer einzigen Variable ist, lassen sich die Vorhersagen des Modells leicht als Funktion der Eingabe betrachten:

In [ ]:
# Wir sagen MPG für 251 verschiedene PS-Werte vorher
x = tf.linspace(0.0, 250, 251)
y = horsepower_model.predict(x)

In [ ]:
def plot_horsepower(x, y):
    plt.scatter(X_train["Horsepower"], y_train, label="Data")
    plt.plot(x, y, color="k", label="Predictions")
    plt.xlabel("Horsepower")
    plt.ylabel("MPG")
    plt.legend()

In [ ]:
plot_horsepower(x, y)

### Mehrere Eingaben

Mit einem fast identischen Setup kannst du Vorhersagen auf Basis mehrerer Eingaben treffen. Dieses Modell rechnet immer noch dasselbe $y = mx+b$, nur dass $m$ eine Matrix und $b$ ein Vektor ist.

Dieses Mal verwenden wir die `Normalization`-Schicht, die an den gesamten Datensatz angepasst wurde.

In [ ]:
linear_model = tf.keras.Sequential([normalizer, layers.Dense(units=1)])

Wenn du dieses Modell mit einem Batch von Eingaben aufrufst, erzeugt es für jedes Beispiel `units=1` Ausgaben.

In [ ]:
linear_model.predict(X_train[:10] * 1.0)

Wenn du das Modell aufrufst, werden seine Gewichtsmatrizen erstellt. Jetzt siehst du, dass der `kernel` (das $m$ in $y=mx+b$) die Form `(9,1)` hat.

In [ ]:
linear_model.layers[1].kernel

Verwende dieselben Aufrufe von `compile` und `fit` wie beim `horsepower`-Modell mit einer Eingabe:

In [ ]:
linear_model.compile(optimizer=Adam(learning_rate=0.1), loss=["mae"], metrics=["mse"])

In [ ]:
%%time
history = linear_model.fit(
    X_train * 1.0,
    y_train,
    epochs=100,
    # Logging unterdrücken (auch hier: Ändere den Wert auf 1, wenn du mehr Informationen zu jeder Epoche ausgeben willst)
    verbose=0,
    # Validierungsergebnisse auf 20% der Trainingsdaten berechnen
    validation_split=0.2,
    # Informationen für TensorBoard speichern
    callbacks=get_callbacks("linear_model"),
)

Mit allen Eingaben erreicht das Modell einen viel niedrigeren Trainings- und Validierungsfehler als das `horsepower`-Modell: 

In [ ]:
plot_loss(history)

Wir sammeln das Ergebnis auf dem Testdatensatz für später in unserem zuvor definierten Dictionary:

In [ ]:
test_results["linear_model"] = linear_model.evaluate(
    np.array(X_test).astype("float32"), y_test, verbose=0
)

## Eine DNN-Regression

Im vorherigen Abschnitt wurden lineare Modelle für eine und für mehrere Eingaben implementiert.

In diesem Abschnitt werden DNN-Modelle mit einer und mit mehreren Eingaben implementiert. Der Code ist im Grunde derselbe, nur wird das Modell um einige „verborgene“ nichtlineare Schichten erweitert. „Verborgen“ bedeutet hier nur, dass sie nicht direkt mit den Eingaben oder Ausgaben verbunden sind.

Diese Modelle enthalten ein paar Schichten mehr als das lineare Modell:

1. Die Normalisierungsschicht.
2. Zwei verborgene, nichtlineare `Dense`-Schichten mit der Nichtlinearität `gelu`.
3. Eine lineare Ausgabeschicht mit einer einzigen Ausgabe.

Beide verwenden dasselbe Trainingsverfahren, deshalb nehmen wir die Methode `compile` in die unten neu definierte Funktion `build_and_compile_model` auf.

In [ ]:
def build_and_compile_model(norm):
    model = keras.Sequential(
        [
            norm,
            layers.Dense(64, activation="gelu"),
            layers.Dense(64, activation="gelu"),
            layers.Dense(1),
        ]
    )

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=["mae"],
        metrics=["mse"],
    )
    return model

### Eine Variable

Wir beginnen wieder mit einem DNN-Modell für eine einzige Eingabe: "Horsepower".

Wir können die Normalisierungsschicht verwenden, die wir für unser erstes Modell erstellt haben. 

In [ ]:
dnn_horsepower_model = build_and_compile_model(horsepower_normalizer)

Dieses Modell hat deutlich mehr trainierbare Parameter als die linearen Modelle.

In [ ]:
dnn_horsepower_model.summary()

Trainieren wir das Modell:

In [ ]:
%%time
history = dnn_horsepower_model.fit(
    X_train["Horsepower"],
    y_train,
    validation_split=0.2,
    verbose=0,
    epochs=100,
    callbacks=get_callbacks("dnn_horsepower"),
)

Dieses Modell schneidet etwas besser ab als das lineare PS-Modell.

In [ ]:
plot_loss(history)

Wenn du die Vorhersagen als Funktion von `Horsepower` darstellst, siehst du, wie dieses Modell die Nichtlinearität der verborgenen Schichten nutzt:

In [ ]:
x = tf.linspace(0.0, 250, 251)
y = dnn_horsepower_model.predict(x)

In [ ]:
plot_horsepower(x, y)

Wir sammeln auch die Ergebnisse auf dem Testdatensatz für später:

In [ ]:
test_results["dnn_horsepower_model"] = dnn_horsepower_model.evaluate(
    X_train["Horsepower"], y_train, verbose=0
)

### Vollständiges Modell

Wenn du diesen Vorgang mit allen Eingaben wiederholst, verbessert sich die Leistung auf dem Validierungsdatensatz leicht.

In [ ]:
dnn_model = build_and_compile_model(normalizer)
dnn_model.summary()

In [ ]:
%%time
history = dnn_model.fit(
    X_train * 1.0,
    y_train,
    validation_split=0.2,
    verbose=0,
    epochs=100,
    callbacks=get_callbacks("dnn_model"),
)

In [ ]:
plot_loss(history)

Sammle die Ergebnisse auf dem Testdatensatz:

In [ ]:
test_results["dnn_model"] = dnn_model.evaluate(X_test, y_test, verbose=0)

## Leistung

Jetzt, da alle Modelle trainiert sind, prüfen wir die Leistung auf dem Testdatensatz und vergleichen, wie gut sie abgeschnitten haben:

In [ ]:
pd.DataFrame(
    test_results, index=["Mean absolute error [MPG]", "Mean squared error [MPG]"]
).T

Diese Ergebnisse passen zum Validierungsfehler, der während des Trainings zu sehen war.

### Vorhersagen treffen

Erstell zum Schluss Vorhersagen für den Testdatensatz und sieh dir die Fehler an, die das Modell dabei macht:

In [ ]:
y_pred = dnn_model.predict(X_test).flatten()

a = plt.axes(aspect="equal")
plt.scatter(y_test, y_pred)
plt.xlabel("True Values [MPG]")
plt.ylabel("Predictions [MPG]")
lims = [0, 50]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims, lims)

Das Modell scheint recht gut vorherzusagen.

Sieh dir jetzt die Verteilung der Fehler an:

In [ ]:
error = y_pred - y_test
plt.hist(error, bins=25)
plt.xlabel("Prediction Error [MPG]")
_ = plt.ylabel("Count")

Wenn du mit dem Modell zufrieden bist, kannst du es für später speichern:

In [ ]:
dnn_model.save("dnn_model.keras")

Wenn du das Modell neu lädst, liefert es identische Ausgaben:

In [ ]:
reloaded = tf.keras.models.load_model("dnn_model.keras")

test_results["reloaded"] = reloaded.evaluate(X_test, y_test, verbose=0)

In [ ]:
pd.DataFrame(
    test_results, index=["Mean absolute error [MPG]", "Mean squared error [MPG]"]
).T

## Fazit

Dieses Notebook hat einige Techniken für ein Regressionsproblem vorgestellt. Hier sind noch ein paar Tipps, die helfen können:

* Der [mittlere quadratische Fehler (MSE)](https://www.tensorflow.org/api_docs/python/tf/losses/MeanSquaredError) und der [mittlere absolute Fehler (MAE)](https://www.tensorflow.org/api_docs/python/tf/losses/MeanAbsoluteError) sind gängige Verlustfunktionen für Regressionsprobleme. Der mittlere absolute Fehler ist weniger empfindlich gegenüber Ausreißern. Für Klassifikationsprobleme werden andere Verlustfunktionen verwendet.
* Ebenso unterscheiden sich die Evaluationsmetriken für die Regression von denen für die Klassifikation.
* Wenn numerische Features der Eingabedaten Werte in unterschiedlichen Wertebereichen haben, sollte jedes Feature unabhängig auf denselben Wertebereich skaliert werden.
* Overfitting ist ein häufiges Problem bei DNN-Modellen, in diesem Tutorial war es keins. Mehr Hilfe dazu findest du im Tutorial [overfit and underfit](https://www.tensorflow.org/tutorials/keras/overfit_and_underfit).


---
## TensorBoard

Im Machine Learning musst du etwas oft messen können, um es zu verbessern. TensorBoard ist ein Tool, das die Messungen und Visualisierungen liefert, die du im Machine-Learning-Workflow brauchst. Damit kannst du Experiment-Metriken wie Verlust und Accuracy nachverfolgen, den Modellgraphen visualisieren, Embeddings in einen niedrigdimensionalen Raum projizieren und vieles mehr.

TensorBoard lässt sich direkt in einem Notebook in Colab und Jupyter verwenden. Das kann hilfreich sein, um Ergebnisse zu teilen, TensorBoard in bestehende Workflows zu integrieren und TensorBoard zu nutzen, ohne lokal etwas zu installieren.

Ein gutes einführendes Notebook findest du [hier](https://www.tensorflow.org/tensorboard/get_started).

Wenn du dieses Notebook in einem Browser ausführst und das TensorBoard nicht angezeigt werden kann: Falls du Safari verwendest, versuch, zu Google Chrome zu wechseln und es noch einmal auszuführen.

### Beispiel mit TensorBoard

Wenn du beim Training mit `.fit()` von Keras den Callback `tf.keras.callbacks.TensorBoard` hinzufügst, werden Logs erstellt und gespeichert. Aktiviere außerdem mit histogram_freq=1 die Berechnung von Histogrammen in jeder Epoche (standardmäßig ist sie ausgeschaltet).

Wir haben die Logs in einem Unterverzeichnis mit Zeitstempel gespeichert, damit sich verschiedene Trainingsdurchläufe leicht auswählen lassen.

In [ ]:
# Notebook-Erweiterung für TensorBoard laden
%reload_ext tensorboard

Mit Magics kannst du TensorBoard im Notebook starten. Am Ende des Befehls musst du den Pfad angeben, unter dem die Log-Dateien gespeichert sind.

In [ ]:
%tensorboard --logdir=./my_logs

Das TensorBoard ist unter http://localhost:6006/ erreichbar